### Import Dependencies

In [76]:
import openai
from pydantic import BaseModel, Field
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
import os

from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery, MatchAny

import numpy as np
import tiktoken
from dotenv import load_dotenv
load_dotenv("../../.env")


True

### Qdrant Client for Qdrant Cloud

In [77]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_CLOUD_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=120
)

## Items Collection

### Create Qdrant colection for hybrid search

In [3]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.20.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


True

In [4]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [5]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [6]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [7]:
df_items = pd.read_json("../../data/raw/meta_Electronics_2022_2023_ratings_100_sample_1000.jsonl", lines=True)

In [8]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,[Specifically design for Samsung Galaxy Tab E ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,[Ultimate Ears boom 2 is the 360-degree wirele...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,"[Product Description, The Seagate GoFlex TV HD...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN


### Preprocess title and features

In [9]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [10]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [11]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [12]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Computers,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,4.5,153,[Specifically Designed For: Samsung Galaxy Tab...,Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lig...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Leafbook,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '7 Inches', '...",B01DJI7TP8,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51PML408lo...
1,All Electronics,Lume Cube AIR Video Conference Lighting Kit - ...,3.7,215,[],Lume Cube AIR Video Conference Lighting Kit - ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],LUME CUBE,"[Electronics, Camera & Photo, Lighting & Studi...",{'Product Dimensions': '1.61 x 1.61 x 1.1 inch...,B07M6SWKLT,NaN,NaN,NaN,https://m.media-amazon.com/images/I/411wKstRw1...
2,All Electronics,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,4.5,1278,[Insanely good 360˚ sound - Ultimate Ears Boom...,Ultimate Ears BOOM 2 Portable Waterproof & Sho...,68.49,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'HONEST review of the UE Boom 2 spe...,Ultimate Ears,"[Electronics, Portable Audio & Video, Portable...",{'Product Dimensions': '2.75 x 2.75 x 7.08 inc...,B07J31M7PW,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41kYEBxNW2...
3,All Electronics,Seagate FreeAgent GoFlex TV STAJ100,3.5,108,[Watch streaming media directly from the Inter...,Seagate FreeAgent GoFlex TV STAJ100 Watch stre...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Seagate,"[Electronics, Television & Video, Streaming Me...","{'Brand Name': 'Seagate', 'Item Weight': '10.7...",B003IT6YFK,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31Vy2ogKvD...
4,All Electronics,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",4.5,110,[🎧【High-quality Stereo Sound】Built-in multiple...,"JSAUX RCA Cable（10ft/3M, RCA Audio Cable Shiel...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'J&D 2RCA Male to 2RCA Male Cable',...",JSAUX,"[Electronics, Television & Video, Accessories,...",{'Package Dimensions': '10.16 x 3.15 x 0.71 in...,B099ZPY68J,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41EbtDHhNE...


In [13]:
list(df_items["description"].items())[0]

(0,
 "Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lightweight Shell Case for Samsung Galaxy Tab E lite 7.0 & Tab 3 Lite T110/T111, Black Specifically Designed For: Samsung Galaxy Tab 3 Lite 7.0 7'' T110 T111 Tablet / Galaxy Tab E Lite 7.0 (NOT FIT FOR Samsung Galaxy Tab 3 7.0 T210) Premium PU leather exterior and microfiber interior. Ultra slim and lightweight hard back design adds minimal bulk while protecting your precious device. Precise cut-outs and openings for easy access to all tablet features. Cover has flip capability to transform the case into a viewing stand and keyboard stand. One piece case, the front and back does not separate. Various colors are available.Magnetic closure.")

In [14]:
list(df_items["image"].items())[0]

(0, 'https://m.media-amazon.com/images/I/51PML408loL._AC_.jpg')

In [15]:
data_to_embed = df_items[["description", "image", "rating_number", "price", "average_rating", "parent_asin"]].to_dict(orient="records")

In [16]:
data_to_embed

[{'description': "Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lightweight Shell Case for Samsung Galaxy Tab E lite 7.0 & Tab 3 Lite T110/T111, Black Specifically Designed For: Samsung Galaxy Tab 3 Lite 7.0 7'' T110 T111 Tablet / Galaxy Tab E Lite 7.0 (NOT FIT FOR Samsung Galaxy Tab 3 7.0 T210) Premium PU leather exterior and microfiber interior. Ultra slim and lightweight hard back design adds minimal bulk while protecting your precious device. Precise cut-outs and openings for easy access to all tablet features. Cover has flip capability to transform the case into a viewing stand and keyboard stand. One piece case, the front and back does not separate. Various colors are available.Magnetic closure.",
  'image': 'https://m.media-amazon.com/images/I/51PML408loL._AC_.jpg',
  'rating_number': 153,
  'price': nan,
  'average_rating': 4.5,
  'parent_asin': 'B01DJI7TP8'},
 {'description': 'Lume Cube AIR Video Conference Lighting Kit - for Video Conference, Streaming, and Remote Working Magnet

In [17]:
len(data_to_embed)

1000

In [18]:
text_to_embed = [item["description"] for item in data_to_embed]

In [19]:
text_to_embed

["Galaxy Tab E Lite 7.0 Case - Leafbook Slim Lightweight Shell Case for Samsung Galaxy Tab E lite 7.0 & Tab 3 Lite T110/T111, Black Specifically Designed For: Samsung Galaxy Tab 3 Lite 7.0 7'' T110 T111 Tablet / Galaxy Tab E Lite 7.0 (NOT FIT FOR Samsung Galaxy Tab 3 7.0 T210) Premium PU leather exterior and microfiber interior. Ultra slim and lightweight hard back design adds minimal bulk while protecting your precious device. Precise cut-outs and openings for easy access to all tablet features. Cover has flip capability to transform the case into a viewing stand and keyboard stand. One piece case, the front and back does not separate. Various colors are available.Magnetic closure.",
 'Lume Cube AIR Video Conference Lighting Kit - for Video Conference, Streaming, and Remote Working Magnetic LED Bundle with Suction Cup Mount ',
 "Ultimate Ears BOOM 2 Portable Waterproof & Shockproof Bluetooth Speaker - Phantom Edition Insanely good 360˚ sound - Ultimate Ears Boom 2 is the 360-degree wi

In [20]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [21]:
len(embeddings)

1000

In [22]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i += 1

In [23]:
pointstructs[0].vector

{'text-embedding-3-small': [0.027679443359375,
  -0.061859130859375,
  -0.01552581787109375,
  0.01210784912109375,
  -0.0174102783203125,
  -0.0279083251953125,
  0.0120391845703125,
  0.040191650390625,
  0.0382080078125,
  -0.02239990234375,
  -0.030303955078125,
  -0.0184783935546875,
  -0.0175018310546875,
  0.02581787109375,
  0.00899505615234375,
  0.031280517578125,
  0.0712890625,
  -0.0204925537109375,
  -0.02447509765625,
  0.0631103515625,
  -0.0033168792724609375,
  0.01027679443359375,
  0.03973388671875,
  0.0067138671875,
  -0.0145263671875,
  -0.04595947265625,
  0.006336212158203125,
  -0.037017822265625,
  0.00232696533203125,
  0.0111083984375,
  0.040679931640625,
  -0.016998291015625,
  0.032470703125,
  -0.07818603515625,
  -0.049835205078125,
  -0.00176239013671875,
  0.0362548828125,
  -0.038726806640625,
  -0.01500701904296875,
  0.04595947265625,
  -0.0005125999450683594,
  0.0462646484375,
  0.03558349609375,
  0.0180816650390625,
  0.00765228271484375,
  -0

In [26]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [27]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [28]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [29]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

In [30]:
results

{'retrieved_context_ids': ['B07Q38R2XF',
  'B08Z3N1YTR',
  'B07L8HX46F',
  'B08DNC3L4M',
  'B0BGHMDZHG',
  'B07ZNPPXN6',
  'B00Y1NGRM2',
  'B0788JXFXB',
  'B07KFXQFQ6',
  'B004M5H6D8',
  'B00B3507Q8',
  'B081122SFR',
  'B073R8PCB2',
  'B09KY4D7RJ',
  'B08DMY7S84',
  'B07RW5XZSH',
  'B0BNJ1QQCM',
  'B09G2NKNW4',
  'B0829D5423',
  'B08HRVCLXL'],
 'retrieved_context': ['ASURION 2 Year Tablet Accident Protection Plan ($30 - $39.99) NO ADDITIONAL COST: You pay $0 for repairs – parts, labor and shipping included. COVERAGE: Plan starts on the date of purchase. Drops, spills and cracked screens due to normal use are covered from day one. Malfunctions are covered after the manufacturer’s warranty. EASY CLAIMS PROCESS: File a claim anytime online at www.Asurion.com/Amazon or by phone. Most claims approved within minutes. We will send you an Amazon e-gift card for the purchase price of your covered product. In some cases, we will replace or repair it. EXPERT TECH HELP: Real experts are available 

### Hybrid Search with weighted RRF

In [31]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [32]:
results = retrieve_data("Can I get a tablet?", qdrant_client, k=20)

In [33]:
results

{'retrieved_context_ids': ['B07Q38R2XF',
  'B07L8HX46F',
  'B0BGHMDZHG',
  'B08Z3N1YTR',
  'B00Y1NGRM2',
  'B004M5H6D8',
  'B081122SFR',
  'B08DNC3L4M',
  'B08DMY7S84',
  'B00B3507Q8',
  'B07RW5XZSH',
  'B07ZNPPXN6',
  'B0829D5423',
  'B08C2DVS1V',
  'B0788JXFXB',
  'B07YQW55CN',
  'B07FSHNRXX',
  'B0128Z5BPI',
  'B07KFXQFQ6',
  'B08Y55D9JM'],
 'retrieved_context': ['ASURION 2 Year Tablet Accident Protection Plan ($30 - $39.99) NO ADDITIONAL COST: You pay $0 for repairs – parts, labor and shipping included. COVERAGE: Plan starts on the date of purchase. Drops, spills and cracked screens due to normal use are covered from day one. Malfunctions are covered after the manufacturer’s warranty. EASY CLAIMS PROCESS: File a claim anytime online at www.Asurion.com/Amazon or by phone. Most claims approved within minutes. We will send you an Amazon e-gift card for the purchase price of your covered product. In some cases, we will replace or repair it. EXPERT TECH HELP: Real experts are available 

## Reviews Collection

In [34]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    }
)

True

In [35]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [36]:
def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [37]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Retrieve all item IDs from Amazon Items Qdrant Collection

In [38]:
dummy_vector = np.zeros(1536).tolist()

In [39]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [40]:
payload.points

[ScoredPoint(id=405, version=3, score=0.0, payload={'parent_asin': 'B07XDPXH95'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=234, version=3, score=0.0, payload={'parent_asin': 'B071CJNFJ2'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=349, version=3, score=0.0, payload={'parent_asin': 'B00CXUT5ZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=271, version=3, score=0.0, payload={'parent_asin': 'B004S472EQ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=147, version=3, score=0.0, payload={'parent_asin': 'B08K3F55GY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=352, version=3, score=0.0, payload={'parent_asin': 'B07JLL7MZ5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=376, version=3, score=0.0, payload={'parent_asin': 'B0741CXH71'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=416, version=3, score=0.0, payload={'parent_asin': 'B0BMZ7V5L1'}, vector=N

In [41]:
len(payload.points)

1000

In [42]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [43]:
parent_asin_list

['B07XDPXH95',
 'B071CJNFJ2',
 'B00CXUT5ZG',
 'B004S472EQ',
 'B08K3F55GY',
 'B07JLL7MZ5',
 'B0741CXH71',
 'B0BMZ7V5L1',
 'B07TSD6PB1',
 'B08HK5XCQ4',
 'B083V2W4SZ',
 'B01DJI7TP8',
 'B07Y1T6NWT',
 'B08D6TVVY5',
 'B07Y5LZBK4',
 'B07YFPGGM5',
 'B09HTKXXPT',
 'B00PNGM8UQ',
 'B0947572HS',
 'B08P4S4S4C',
 'B01FEDC0M8',
 'B08X155BYB',
 'B08PBM1TR4',
 'B09MCWWLBC',
 'B0182UE3I4',
 'B009XP1J26',
 'B085KVH16T',
 'B08H8MJY27',
 'B07HZ636L8',
 'B08Z44LXYL',
 'B089QBHN73',
 'B06XGCJLR6',
 'B078NNFHQ7',
 'B09M7Q8N8R',
 'B07NZNMLYC',
 'B07YY1N5XQ',
 'B08356M6N7',
 'B09HX83S9W',
 'B07MT4VJRN',
 'B01FAR9OQI',
 'B0001M2FYQ',
 'B08DNC3L4M',
 'B076T8L7Q2',
 'B0055PADN4',
 'B01N2NKI27',
 'B00HQIN4DY',
 'B08C2DVS1V',
 'B088JN2GGQ',
 'B003IT6YFK',
 'B0978LG2ZJ',
 'B076VJDWBP',
 'B00B1GVD5I',
 'B000HWQ75I',
 'B07BXJ6KH6',
 'B08T5SXW61',
 'B07D5JYGDP',
 'B07CNT5ST1',
 'B0C54BYRTF',
 'B09JVNM8PP',
 'B07ZYYYPQN',
 'B009VJLZ14',
 'B084LMFXF1',
 'B07MJPMQCH',
 'B00A7IZNNO',
 'B01CVHARWU',
 'B01DFKANQM',
 'B0C3VV6D

### Load Amazon Reviews Dataset

In [44]:
df_reviews = pd.read_json("../../data/raw/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [45]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Love it! It fits my Chromebook perfectly,Love it! It fits my Chromebook perfectly.,[],B00J39BYSG,B07BJ8KD6X,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-06-30 21:27:51.000,0,True
1,5,love this,i was concerned it wasnt gonna work with my TV...,[],B008GVOVK0,B008GVOVK0,AHGAOIZVODNHYMNCBV4DECZH42UQ,2018-08-24 22:21:36.983,0,True
2,5,nice product,adjustable and elastic,[],B01G54B68Y,B07HMRN7Y6,AGEP2CC2QIJU5IHQZXCC7BE5JSDQ,2019-11-12 17:33:07.317,0,True
3,4,Satisfied,Works well,[],B019WJROSK,B019WJROSK,AHSAUHUTLVNKJ72GUIPGI5XZH6ZA,2019-02-01 20:54:53.408,0,True
4,5,Works great,This wire works great. I recommend it. It's ...,[],B08G8PMN8C,B08J42GSW6,AHATA6X6MYTC3VNBFJ3WIYVK257A,2021-02-25 01:59:27.685,0,False


In [46]:
len(df_reviews)

137987

In [47]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [48]:
len(df_reviews_sample)

137987

### Define functions to preprocess reviews data

In [49]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [50]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [51]:
encoding.encode("Can I get some earphones?")

[6854, 358, 636, 1063, 2487, 17144, 30]

In [52]:
def token_count(row, model="text-embedding-3-small"):

    encoding = tiktoken.encoding_for_model(model)

    return len(encoding.encode(row["preprocessed_data"]))

In [53]:
df_reviews_sample["preprocessed_data"] = df_reviews_sample.apply(preprocess_reviews_data, axis=1)
df_reviews_sample["token_count"] = df_reviews_sample.apply(token_count, axis=1)

In [54]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,token_count
0,5,Love it! It fits my Chromebook perfectly,Love it! It fits my Chromebook perfectly.,[],B00J39BYSG,B07BJ8KD6X,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,2015-06-30 21:27:51.000,0,True,Love it! It fits my Chromebook perfectly Love ...,20
1,5,love this,i was concerned it wasnt gonna work with my TV...,[],B008GVOVK0,B008GVOVK0,AHGAOIZVODNHYMNCBV4DECZH42UQ,2018-08-24 22:21:36.983,0,True,love this i was concerned it wasnt gonna work ...,94
2,5,nice product,adjustable and elastic,[],B01G54B68Y,B07HMRN7Y6,AGEP2CC2QIJU5IHQZXCC7BE5JSDQ,2019-11-12 17:33:07.317,0,True,nice product adjustable and elastic,5
3,4,Satisfied,Works well,[],B019WJROSK,B019WJROSK,AHSAUHUTLVNKJ72GUIPGI5XZH6ZA,2019-02-01 20:54:53.408,0,True,Satisfied Works well,4
4,5,Works great,This wire works great. I recommend it. It's ...,[],B08G8PMN8C,B08J42GSW6,AHATA6X6MYTC3VNBFJ3WIYVK257A,2021-02-25 01:59:27.685,0,False,Works great This wire works great. I recommen...,21


In [55]:
len(df_reviews_sample)

137987

In [56]:
df_reviews_sample = df_reviews_sample[df_reviews_sample["token_count"] < 8192]

In [57]:
len(df_reviews_sample)

137987

In [58]:
total_tokens = df_reviews_sample["token_count"].sum()

In [59]:
total_tokens

np.int64(8247850)

### Embed the text and add additional filds to the payload of each vector for reviews

In [60]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [61]:
data_to_embed_reviews

[{'preprocessed_data': 'Love it! It fits my Chromebook perfectly Love it!  It fits my Chromebook perfectly.',
  'parent_asin': 'B07BJ8KD6X'},
 {'preprocessed_data': 'love this i was concerned it wasnt gonna work with my TV and firestick cuz the tv is from 2012 and i only had one HDMI port ... i read reviews some said it worked - some said not ... so for the price i took a chance and it WORKED fine  and without issues ... finally something worked without issues  just plugged it in !! hahaha  and i still have a 3rd HDMI port if i need it in the future !',
  'parent_asin': 'B008GVOVK0'},
 {'preprocessed_data': 'nice product adjustable and elastic',
  'parent_asin': 'B07HMRN7Y6'},
 {'preprocessed_data': 'Satisfied Works well', 'parent_asin': 'B019WJROSK'},
 {'preprocessed_data': "Works great This wire works great.  I recommend it.  It's well made and good quality.",
  'parent_asin': 'B08J42GSW6'},
 {'preprocessed_data': 'Five Stars Very good quality!',
  'parent_asin': 'B09V6JXVX1'},
 {'pr

In [62]:
len(data_to_embed_reviews)

137987

In [63]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [64]:
text_to_embed_reviews

['Love it! It fits my Chromebook perfectly Love it!  It fits my Chromebook perfectly.',
 'love this i was concerned it wasnt gonna work with my TV and firestick cuz the tv is from 2012 and i only had one HDMI port ... i read reviews some said it worked - some said not ... so for the price i took a chance and it WORKED fine  and without issues ... finally something worked without issues  just plugged it in !! hahaha  and i still have a 3rd HDMI port if i need it in the future !',
 'nice product adjustable and elastic',
 'Satisfied Works well',
 "Works great This wire works great.  I recommend it.  It's well made and good quality.",
 'Five Stars Very good quality!',
 'Excellent cable Perfect length with room to spare. I really like they these cables come flat, I can easily tuck under a rug or carpet to hide them, which makes them less of a trip hazard. Excellent speeds from this Cat 8 cable.',
 "Nice product. Works well.<br /><br />Pros:<br />Paired easily with iPhone 6<br />Cables are p

In [65]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 137987
Processed 1000 of 137987
Processed 1500 of 137987
Processed 2000 of 137987
Processed 2500 of 137987
Processed 3000 of 137987
Processed 3500 of 137987
Processed 4000 of 137987
Processed 4500 of 137987
Processed 5000 of 137987
Processed 5500 of 137987
Processed 6000 of 137987
Processed 6500 of 137987
Processed 7000 of 137987
Processed 7500 of 137987
Processed 8000 of 137987
Processed 8500 of 137987
Processed 9000 of 137987
Processed 9500 of 137987
Processed 10000 of 137987
Processed 10500 of 137987
Processed 11000 of 137987
Processed 11500 of 137987
Processed 12000 of 137987
Processed 12500 of 137987
Processed 13000 of 137987
Processed 13500 of 137987
Processed 14000 of 137987
Processed 14500 of 137987
Processed 15000 of 137987
Processed 15500 of 137987
Processed 16000 of 137987
Processed 16500 of 137987
Processed 17000 of 137987
Processed 17500 of 137987
Processed 18000 of 137987
Processed 18500 of 137987
Processed 19000 of 137987
Processed 19500 of 137987
Proces

In [66]:
len(embeddings)

137987

In [67]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
            },
            payload=data
        )
    )
    i += 1

In [68]:
pointstructs[0].vector

{'text-embedding-3-small': [0.03839111328125,
  -0.040618896484375,
  -0.037139892578125,
  -0.0289459228515625,
  -0.0009937286376953125,
  -0.07733154296875,
  -0.01029205322265625,
  0.0008897781372070312,
  -0.046234130859375,
  0.0026760101318359375,
  -0.024322509765625,
  0.0256500244140625,
  -0.0278472900390625,
  0.007015228271484375,
  0.04730224609375,
  0.002651214599609375,
  -0.0382080078125,
  -0.0237884521484375,
  0.0204620361328125,
  0.0257415771484375,
  -0.01316070556640625,
  0.03973388671875,
  0.01580810546875,
  -0.03167724609375,
  0.00475311279296875,
  -0.050811767578125,
  -0.028045654296875,
  0.0034694671630859375,
  0.0203704833984375,
  -0.054931640625,
  -0.007251739501953125,
  -0.0215301513671875,
  -0.00521087646484375,
  -0.052490234375,
  -0.01488494873046875,
  -0.01050567626953125,
  -0.0094146728515625,
  -0.026580810546875,
  -0.01154327392578125,
  0.0489501953125,
  -0.0217742919921875,
  0.055511474609375,
  0.0880126953125,
  0.0499877929

In [78]:
batch_size_qdrant = 200
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

Processed 200 of 137987
Processed 400 of 137987
Processed 600 of 137987
Processed 800 of 137987
Processed 1000 of 137987
Processed 1200 of 137987
Processed 1400 of 137987
Processed 1600 of 137987
Processed 1800 of 137987
Processed 2000 of 137987
Processed 2200 of 137987
Processed 2400 of 137987
Processed 2600 of 137987
Processed 2800 of 137987
Processed 3000 of 137987
Processed 3200 of 137987
Processed 3400 of 137987
Processed 3600 of 137987
Processed 3800 of 137987
Processed 4000 of 137987
Processed 4200 of 137987
Processed 4400 of 137987
Processed 4600 of 137987
Processed 4800 of 137987
Processed 5000 of 137987
Processed 5200 of 137987
Processed 5400 of 137987
Processed 5600 of 137987
Processed 5800 of 137987
Processed 6000 of 137987
Processed 6200 of 137987
Processed 6400 of 137987
Processed 6600 of 137987
Processed 6800 of 137987
Processed 7000 of 137987
Processed 7200 of 137987
Processed 7400 of 137987
Processed 7600 of 137987
Processed 7800 of 137987
Processed 8000 of 137987
Proc

In [ ]:
import time
import random

from qdrant_client.http.exceptions import ResponseHandlingException, UnexpectedResponse

batch_size_qdrant = 100
max_retries = 5
base_delay = 1.0

for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]

    for attempt in range(max_retries):
        try:
            qdrant_client.upsert(
                collection_name="Amazon-reviews-collection-01",
                points=batch,
                wait=True,
            )
            break
        except (ResponseHandlingException, UnexpectedResponse, TimeoutError) as e:
            if attempt == max_retries - 1:
                raise RuntimeError(
                    f"Failed to upsert batch starting at index {i} after {max_retries} attempts"
                ) from e
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"Batch at {i} failed ({type(e).__name__}: {e}); retrying in {delay:.1f}s")
            time.sleep(delay)

    print(f"Processed {min(i + batch_size_qdrant, len(pointstructs))} of {len(pointstructs)}")